In [5]:
import json
from dataclasses import dataclass
from typing import Dict, List, Any, Optional

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from datasets import load_dataset
from tqdm import tqdm
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B"
IGNORE_INDEX = -100


# ----------------------------
# 1) Prompt format
# ----------------------------
def build_prompt(source: str) -> str:
    # Adjust this prompt to your task.
    # Example: Java -> C# code translation
    return (
        f"{source}"
    )


# ----------------------------
# 2) Preprocess one example
# ----------------------------
def preprocess_example(
    example: Dict[str, str],
    tokenizer,
    max_length: int = 1024,
) -> Dict[str, List[int]]:
    """
    Training example format:
      full_text = prompt + target + eos

    labels:
      [-100 for prompt tokens] + [target tokens + eos]
    """
    prompt_text = build_prompt(example["source"])
    target_text = example["target"]

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    target_ids = tokenizer(
        target_text,
        add_special_tokens=False,
    )["input_ids"]

    # Add EOS to the answer.
    if tokenizer.eos_token_id is not None:
        target_ids = target_ids + [tokenizer.eos_token_id]

    input_ids = prompt_ids + target_ids
    attention_mask = [1] * len(input_ids)
    labels = [IGNORE_INDEX] * len(prompt_ids) + target_ids.copy()

    # Truncate from the right for training examples
    input_ids = input_ids[:max_length]
    attention_mask = attention_mask[:max_length]
    labels = labels[:max_length]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


# ----------------------------
# 3) Dataset
# ----------------------------
class Seq2SeqStyleCausalLMDataset(Dataset):
    """
    Expects a list of dicts like:
      {"source": "...", "target": "..."}
    """
    def __init__(self, data: List[Dict[str, str]], tokenizer, max_length: int = 1024):
        self.examples = [
            preprocess_example(ex, tokenizer, max_length=max_length)
            for ex in data
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx: int) -> Dict[str, List[int]]:
        return self.examples[idx]


# ----------------------------
# 4) Training collator
#    Right padding for training
# ----------------------------
@dataclass
class TrainCollator:
    tokenizer: Any
    label_pad_token_id: int = IGNORE_INDEX

    def __call__(self, features: List[Dict[str, List[int]]]) -> Dict[str, torch.Tensor]:
        self.tokenizer.padding_side = "right"

        batch = self.tokenizer.pad(
            {
                "input_ids": [f["input_ids"] for f in features],
                "attention_mask": [f["attention_mask"] for f in features],
            },
            padding=True,
            return_tensors="pt",
        )

        max_len = batch["input_ids"].shape[1]
        padded_labels = []
        for f in features:
            y = f["labels"]
            padded_y = y + [self.label_pad_token_id] * (max_len - len(y))
            padded_labels.append(padded_y)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch


# ----------------------------
# 5) Utility: inspect preprocessing
# ----------------------------
def inspect_processed_example(dataset: Dataset, tokenizer, idx: int = 0) -> None:
    ex = dataset[idx]
    print("=" * 100)
    print("FULL INPUT")
    print(tokenizer.decode(ex["input_ids"], skip_special_tokens=False))

    supervised_ids = [t for t in ex["labels"] if t != IGNORE_INDEX]
    print("=" * 100)
    print("SUPERVISED LABEL TEXT")
    print(tokenizer.decode(supervised_ids, skip_special_tokens=False))
    print("=" * 100)


# ----------------------------
# 6) Train
# ----------------------------
def train(
    model,
    tokenizer,
    train_data: List[Dict[str, str]],
    output_dir: str,
    max_length: int = 1024,
    batch_size: int = 4,
    num_epochs: int = 3,
    learning_rate: float = 2e-5,
    warmup_ratio: float = 0.03,
    grad_accum_steps: int = 1,
    device: Optional[str] = None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.to(device)
    model.train()

    train_dataset = Seq2SeqStyleCausalLMDataset(
        train_data,
        tokenizer=tokenizer,
        max_length=max_length,
    )
    inspect_processed_example(train_dataset, tokenizer, idx=0)

    collator = TrainCollator(tokenizer=tokenizer)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collator,
    )

    optimizer = AdamW(model.parameters(), lr=learning_rate)
    total_steps = (len(train_loader) * num_epochs + grad_accum_steps - 1) // grad_accum_steps
    warmup_steps = int(total_steps * warmup_ratio)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    global_step = 0
    for epoch in range(num_epochs):
        running_loss = 0.0
        optimizer.zero_grad()

        for step, batch in tqdm(enumerate(train_loader)):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss / grad_accum_steps
            loss.backward()

            running_loss += loss.item()

            if (step + 1) % grad_accum_steps == 0:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % 10 == 0:
                    print(
                        f"epoch={epoch+1} step={global_step} "
                        f"loss={(running_loss / 10):.4f}"
                    )
                    running_loss = 0.0

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Saved model and tokenizer to: {output_dir}")


# ----------------------------
# 7) Batched inference
#    Left padding for generation
# ----------------------------
@torch.no_grad()
def generate_predictions(
    model,
    tokenizer,
    sources: List[str],
    max_input_length: int = 1024,
    max_new_tokens: int = 128,
    batch_size: int = 4,
    device: Optional[str] = None,
    do_sample: bool = False,
    temperature: float = 0.2,
    top_p: float = 0.95,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.to(device)
    model.eval()

    all_outputs = []

    for i in range(0, len(sources), batch_size):
        batch_sources = sources[i:i + batch_size]
        prompts = [build_prompt(src) for src in batch_sources]

        # Important for batched decoder-only generation
        tokenizer.padding_side = "left"

        enc = tokenizer(
            prompts,
            padding=True,
            truncation=True,
            max_length=max_input_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        prompt_len = enc["input_ids"].shape[1]
        gen_only = outputs[:, prompt_len:]

        decoded = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
        all_outputs.extend(decoded)

    return all_outputs


# ----------------------------
# 8) Load data
# ----------------------------
def load_jsonl(path: str) -> List[Dict[str, str]]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

def create_codetask_dataset(dataset_name, seed, num_train, num_eval, num_test):
    data_dict = {}
    for split in ['train', 'validation', 'test']:
        ds = load_dataset(
                "dongg18/CODETASK_with_instruction_pool",
                data_files={split: f"{dataset_name}/{split}-*.parquet"},
                split=split,
            )
        ds = ds.remove_columns([c for c in ds.column_names if c not in ('input', 'output')])
        ds = ds.rename_column('input', 'source')
        ds = ds.rename_column('output', 'target')    
        n = num_train if split == 'train' else num_eval if split == 'validation' else num_test
        if int(n) != -1:
            ds = ds.shuffle(seed=seed).select(range(int(n)))
        data_dict[split] = ds

    return data_dict['train'], data_dict['validation'], data_dict['test']


In [6]:
output_dir = "/kaggle/working/qwen25coder05b_lora"

train_data, val_data, test_data = create_codetask_dataset("CodeTrans", 0, 2000, 50, 50)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

# Good training defaults for memory
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

train(
    model=model,
    tokenizer=tokenizer,
    train_data=train_data,
    output_dir=output_dir,
    max_length=320,
    batch_size=1,
    num_epochs=2,
    learning_rate=2e-4,
    grad_accum_steps=8,
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093
FULL INPUT
Rewrite this Java code in C#:
public ApproveSkillResult approveSkill(ApproveSkillRequest request) {request = beforeClientExecution(request);return executeApproveSkill(request);}
public virtual ApproveSkillResponse ApproveSkill(ApproveSkillRequest request){var options = new InvokeOptions();options.RequestMarshaller = ApproveSkillRequestMarshaller.Instance;options.ResponseUnmarshaller = ApproveSkillResponseUnmarshaller.Instance;return Invoke<ApproveSkillResponse>(request, options);}
<|endoftext|>
SUPERVISED LABEL TEXT
public virtual ApproveSkillResponse ApproveSkill(ApproveSkillRequest request){var options = new InvokeOptions();options.RequestMarshaller = ApproveSkillRequestMarshaller.Instance;options.ResponseUnmarshaller = ApproveSkillResponseUnmarshaller.Instance;return Invoke<ApproveSkillResponse>(request, options);}
<|endoftext|>


80it [00:19,  4.35it/s]

epoch=1 step=10 loss=1.4681


160it [00:38,  4.26it/s]

...

1920it [07:34,  4.10it/s]

epoch=2 step=490 loss=0.2671


2000it [07:53,  4.23it/s]

epoch=2 step=500 loss=0.3066


Saved model and tokenizer to: /kaggle/working/qwen25coder05b_lora


In [7]:

# Reload for inference if you want
tokenizer = AutoTokenizer.from_pretrained(output_dir, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    output_dir,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

test_sources = []
test_gt = []
for example in test_data:
    test_sources.append(example['source'])
    test_gt.append(example['target'])

preds = generate_predictions(
    model=model,
    tokenizer=tokenizer,
    sources=test_sources,
    max_input_length=320,
    max_new_tokens=256,
    batch_size=2,
    do_sample=False,
)

for src, gt, pred in zip(test_sources, test_gt, preds):
    print("=" * 100)
    print("SOURCE:")
    print(src)
    print("GROUND-TRUTH:")
    print(gt)
    print("\nPREDICTION:")
    print(pred)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/96 [00:00<?, ?it/s]

SOURCE:
Rewrite the following code in C#:
public Object[] toArray() {return a.clone();}

GROUND-TRUTH:
public override object[] toArray(){return (object[])a.Clone();}


PREDICTION:
public override object[] toArray(){return a.clone();}

SOURCE:
Write the C# version of this Java code:
public List<String> getUndeletedList() {return undeletedList;}

GROUND-TRUTH:
public virtual IList<string> GetUndeletedList(){return undeletedList;}


PREDICTION:
public virtual IList<string> GetUndeletedList(){return undeletedList;}

SOURCE:
Convert this code from Java to C#:
public RenameFaceRequest() {super("CloudPhoto", "2017-07-11", "RenameFace", "cloudphoto");setProtocol(ProtocolType.HTTPS);}

GROUND-TRUTH:
public RenameFaceRequest(): base("CloudPhoto", "2017-07-11", "RenameFace", "cloudphoto", "openAPI"){Protocol = ProtocolType.HTTPS;}


PREDICTION:
public RenameFaceRequest(): base("CloudPhoto", "2017-07-11", "RenameFace", "cloudphoto", "openAPI"){Protocol = ProtocolType.HTTPS;}

SOURCE:
Port the fol

In [8]:

# # Reload for inference if you want
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
# )

# preds = generate_predictions(
#     model=model,
#     tokenizer=tokenizer,
#     sources=test_sources,
#     max_input_length=320,
#     max_new_tokens=256,
#     batch_size=2,
#     do_sample=False,
# )

# for src, gt, pred in zip(test_sources, test_gt, preds):
#     print("=" * 100)
#     print("SOURCE:")
#     print(src)
#     print("GROUND-TRUTH:")
#     print(gt)
#     print("\nPREDICTION:")
#     print(pred)